In [144]:
# market kernel
import pandas as pd

## 병합

In [145]:
file_path = "D:\PP\BC"

market_1_path = "D:\PP\BC\data\og\소상공인시장진흥공단_전통시장현황_20240719.csv"
market_2_path = "D:\PP\BC\data\og\소상공인시장진흥공단_시장_점포수_20230801.csv"
ppl_path = "D:\PP\BC\data\og\지역별(행정동) 성별 연령별 주민등록 인구수_20231031.csv"

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:5: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:5: SyntaxWarning: invalid escape sequence '\P'
C:\Users\legen\AppData\Local\Temp\ipykernel_20208\1000622544.py:1: SyntaxWarning: invalid escape sequence '\P'
  file_path = "D:\PP\BC"
C:\Users\legen\AppData\Local\Temp\ipykernel_20208\1000622544.py:3: SyntaxWarning: invalid escape sequence '\P'
  market_1_path = "D:\PP\BC\data\og\소상공인시장진흥공단_전통시장현황_20240719.csv"
C:\Users\legen\AppData\Local\Temp\ipykernel_20208\1000622544.py:4: SyntaxWarning: invalid escape sequence '\P'
  market_2_path = "D:\PP\BC\data\og\소상공인시장진흥공단_시장_점포수_20230801.csv"
C:\Users\legen\AppData\Local\Temp\ipykernel_20208\1000622544.py:5: SyntaxWarning: invalid escape sequence 

In [146]:
market_1 = pd.read_csv(market_1_path, encoding="cp949")
market_2 = pd.read_csv(market_2_path, encoding="cp949")

In [147]:
def market_combine(market_1, market_2):
    for df in [market_1, market_2]:
        df["시장명"] = df["시장명"].astype(str).str.strip()
        df["시군구"] = df["시군구"].astype(str).str.strip()

    # 복합키 생성
    market_1["key"] = market_1["시장명"] + "|" + market_1["시군구"]
    market_2["key"] = market_2["시장명"] + "|" + market_2["시군구"]

    common_keys = set(market_1["key"]) & set(market_2["key"])

    m1 = market_1[market_1["key"].isin(common_keys)].copy()
    m2 = market_2[market_2["key"].isin(common_keys)].copy()

    m2 = m2.drop(columns=["시장명", "시군구"], errors="ignore")

    m1 = m1.reset_index(drop=True)
    m2 = m2.reset_index(drop=True)

    m1 = m1.sort_values("key").reset_index(drop=True)
    m2 = m2.sort_values("key").reset_index(drop=True)

    m2_concat = m2.drop(columns=["시장명", "시군구"], errors="ignore")
    market_df = pd.concat([m1, m2_concat], axis=1) 
    # 중복 컬럼 제거 
    market_df = market_df.loc[:, ~market_df.columns.duplicated()]

    return market_df

In [148]:
market_df = market_combine(market_1, market_2)

In [149]:
ex_cols = [
    '시장코드',
    '시장 유형',
    'key',
    '순번',
    '주소'
]

ex_cols = [c for c in ex_cols if c in market_df.columns]

market_df = market_df.drop(columns=ex_cols)

In [150]:
sido_map = {
    # 특별시 / 광역시
    '서울': '서울특별시',
    '서울시': '서울특별시',
    '서울특별시': '서울특별시',

    '부산': '부산광역시',
    '부산시': '부산광역시',
    '부산광역시': '부산광역시',

    '대구': '대구광역시',
    '대구시': '대구광역시',
    '대구광역시': '대구광역시',

    '인천': '인천광역시',
    '인천시': '인천광역시',
    '인천광역시': '인천광역시',

    '광주': '광주광역시',
    '광주시': '광주광역시',
    '광주광역시': '광주광역시',

    '대전': '대전광역시',
    '대전시': '대전광역시',
    '대전광역시': '대전광역시',

    '울산': '울산광역시',
    '울산시': '울산광역시',
    '울산광역시': '울산광역시',

    '세종': '세종특별자치시',
    '세종시': '세종특별자치시',
    '세종특별자치시': '세종특별자치시',

    # 도 단위
    '경기': '경기도',
    '경기도': '경기도',

    '충북': '충청북도',
    '충청북도': '충청북도',

    '충남': '충청남도',
    '충청남도': '충청남도',

    '전북': '전라북도',
    '전라북도': '전라북도',

    '전남': '전라남도',
    '전라남도': '전라남도',

    '경북': '경상북도',
    '경상북도': '경상북도',

    '경남': '경상남도',
    '경상남도': '경상남도',

    '강원': '강원특별자치도',
    '강원도': '강원특별자치도',
    '강원특별자치도': '강원특별자치도',

    '제주': '제주특별자치도',
    '제주도': '제주특별자치도',
    '제주특별자치도': '제주특별자치도',
}

market_df['시도'] = (market_df['시도'].astype(str).str.strip().map(sido_map))


In [151]:
import numpy as np
import pandas as pd

# 사업지원 Y/N
market_3 = pd.read_csv(r"D:\PP\BC\data\og\소상공인시장진흥공단_전국 전통시장 지원현황.csv", encoding = 'cp949')

# 상인회
market_4 = pd.read_csv(r"D:\PP\BC\data\og\소상공인시장진흥공단_시장 상인조직_20201231.csv", encoding = 'cp949')

# 배송, 장보기 서비스
market_5 = pd.read_csv(r"D:\PP\BC\data\og\소상공인시장진흥공단_시장 정보_09_28_2021_배송_장보기서비스.csv")

# 취급품목
market_6 = pd.read_csv(r"D:\PP\BC\data\og\전국전통시장표준데이터.csv", encoding = 'cp949')

In [152]:
# 상인회
market_4["has_assoc"] = (
    market_4["상인회 조직형태"].notna() |
    market_4["상인회 명"].notna()
).astype(int)

market_4['join_stores'] = market_4['상인회 가입점포 수 - 점포수']

market_4["assoc_is_legal"] = (
    market_4["상인회 조직형태"] == "법적단체"
).astype(int)

market_4 = market_4[market_4['join_stores'].notna()].reset_index()
market_4 = market_4.rename(columns={'시장-상점가명': '시장명'})

In [153]:
market_4.describe()

,index,순번,상인회 가입점포 수 - 점포수,Unnamed: 9,has_assoc,join_stores,assoc_is_legal
count,1280.000000,1280.000000,1280.000000,0.0,1280.0,1280.000000,1280.000000
mean,700.114063,701.114063,114.636719,NaN,1.0,114.636719,0.902344
std,392.051381,392.051381,214.648587,NaN,0.0,214.648587,0.296965
min,2.000000,3.000000,0.000000,NaN,1.0,0.000000,0.000000
25%,369.750000,370.750000,43.000000,NaN,1.0,43.000000,1.000000
50%,699.500000,700.500000,70.500000,NaN,1.0,70.500000,1.000000
75%,1037.250000,1038.250000,121.000000,NaN,1.0,121.000000,1.000000
max,1400.000000,1401.000000,4735.000000,NaN,1.0,4735.000000,1.000000


In [154]:
# 배송, 장보기 서비스
market_5["delivery"] = (market_5["실시여부 - 1) 배송서비스"] == 1).astype(int)
market_5["grocery"] = (market_5["실시여부 - 2) 장보기서비스"] == 1).astype(int)
market_5 = market_5.rename(columns = {'시장-상점가명': '시장명'})

In [155]:
# 취급품목
TOKENS = [
    "농산물", "축산물", "수산물", "가공식품",
    "의류", "신발", "가정용품", "음식점", "근린생활서비스"
]

# + 분리
def parse_items(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split("+") if t.strip()]

market_6["items_list"] = market_6["취급품목"].apply(parse_items)


In [156]:
for t in TOKENS:
    col = {
        "농산물": "has_agri",
        "축산물": "has_livestock",
        "수산물": "has_fish",
        "가공식품": "has_processed",
        "의류": "has_clothing",
        "신발": "has_shoes",
        "가정용품": "has_homegoods",
        "음식점": "has_restaurant",
        "근린생활서비스": "has_service"
    }[t]
    market_6[col] = market_6["items_list"].apply(lambda lst: int(t in lst))

market_6["fresh_count"] = market_6[["has_agri","has_livestock","has_fish"]].sum(axis=1)
market_6["nonfood_count"] = market_6[["has_clothing","has_shoes","has_homegoods","has_service"]].sum(axis=1)
market_6["item_diversity"] = market_6["items_list"].apply(len)

market_6["is_food_based"] = (
    (market_6["fresh_count"] > 0) |
    (market_6["has_processed"] == 1) |
    (market_6["has_restaurant"] == 1)
).astype(int)

market_6["has_nonfood"] = (
    (market_6["nonfood_count"] > 0)
).astype(int)

In [157]:
market_6['item_diversity'].unique()

array([5, 8, 9, 1, 3, 2, 4, 7, 6])

In [158]:
# 시장 유형 라벨 (설명용)
def market_item_type(row):
    fresh = row["fresh_count"]
    nonfood = row["nonfood_count"]
    rest = row["has_restaurant"]
    serv = row["has_service"]
    proc = row["has_processed"]

    # 서비스형 (서비스가 있고, 판매/먹거리 요소가 약한 경우)
    if serv == 1 and fresh == 0 and proc == 0 and rest == 0:
        return "서비스형"

    # 비식품/전문형 (신선 없음 + 비식품 있음)
    if fresh == 0 and nonfood > 0 and (proc == 0) and (rest == 0):
        return "비식품/전문형"

    # 먹거리/외식형 (음식점 중심)
    if rest == 1 and fresh <= 1 and nonfood == 0:
        return "먹거리/외식형"

    # 생활형(장보기형) (신선 2개 이상 + 비식품 거의 없음)
    if fresh >= 2 and nonfood == 0:
        return "생활형(장보기형)"

    # 복합형 (그 외 대부분: 신선/가공/외식 + 비식품 혼합)
    return "복합형(생활+비식품)"

market_6["market_item_type"] = market_6.apply(market_item_type, axis=1)

In [159]:
import pandas as pd
import numpy as np

# market_df = market_df.copy()

# 시장명 key 정규화(공백/특수공백 제거) — "시장명 값이 일치" 전제라서 최소한만 정리
def _norm_name(s):
    if pd.isna(s):
        return np.nan
    return str(s).strip().replace("\u3000", " ").replace("\xa0", " ")

market_df["시장명_key"] = market_df["시장명"].apply(_norm_name)

In [160]:
# market_3 병합

m3 = market_3.copy()
m3["시장명_key"] = m3["시장명"].apply(_norm_name)

# 지원금액 컬럼이 숫자가 아닐 수도 있으니 안전 변환
# m3["지원금액"] = pd.to_numeric(m3["지원금액"], errors="coerce")

m3_agg = (
    m3.groupby("시장명_key", as_index=False)
      .agg(support_amt=("지원금액", "sum"))
)
m3_agg["지원여부"] = (m3_agg["support_amt"] > 0).astype(int)
m3_agg = m3_agg.rename(columns={'support_amt': "지원금액"})
market_df = market_df.merge(m3_agg, on="시장명_key", how="left")

In [161]:
# market_4 병합

m4 = market_4.copy()
m4["시장명_key"] = m4["시장명"].apply(_norm_name)

m4_keep = m4[["시장명_key", "has_assoc", "join_stores"]].copy()

# 중복 시장명이 있을 수 있으니 시장명 단위로 1행으로 정리(안전)
m4_keep = (
    m4_keep.groupby("시장명_key", as_index=False)
           .agg(has_assoc=("has_assoc", "max"),
                join_stores=("join_stores", "max"))
)

market_df = market_df.merge(m4_keep, on="시장명_key", how="left")

In [162]:
# market_5 병합

m5 = market_5.copy()
m5["시장명_key"] = m5["시장명"].apply(_norm_name)

m5_keep = m5[["시장명_key", "delivery", "grocery"]].copy()

m5_keep = (
    m5_keep.groupby("시장명_key", as_index=False)
           .agg(delivery=("delivery", "max"),
                grocery=("grocery", "max"))
)

market_df = market_df.merge(m5_keep, on="시장명_key", how="left")

In [163]:
import re
import pandas as pd
import numpy as np

def normalize_sido(token: str) -> str:
    if pd.isna(token):
        return np.nan
    s = str(token).strip()

    # 대표적인 별칭/개편 표기 통일
    # (네가 언급한 전북특별자치도 포함)
    mapping = {
        "전북특별자치도": "전라북도",
        "전북": "전라북도",
        "전남": "전라남도",
        "충북": "충청북도",
        "충남": "충청남도",
        "경북": "경상북도",
        "경남": "경상남도",
        # 필요 시 추가
    }
    return mapping.get(s, s)

def get_sido_from_addr(addr: str) -> str:
    if pd.isna(addr):
        return np.nan
    first = str(addr).strip().split()[0]  # 공백 기준 첫 토큰
    return normalize_sido(first)



In [164]:
# market_df 쪽
mdf = market_df.copy()
mdf["시장명_key"] = mdf["시장명"].apply(_norm_name)
mdf["sido_key"] = mdf["지번주소"].apply(get_sido_from_addr)

# market_6 쪽
m6 = market_6.copy()
m6["시장명_key"] = m6["시장명"].apply(_norm_name)
m6["sido_key"] = m6["소재지지번주소"].apply(get_sido_from_addr)


In [165]:
# 붙일 컬럼 선택 (원하는 것만)
add_cols = [
    "item_diversity",
    "is_food_based",
    "has_nonfood",
    "fresh_count",
    "nonfood_count",
    'market_item_type'
    # 필요하면 has_* 더미도 추가 가능
]

m6_keep = m6[["시장명_key", "sido_key"] + add_cols].copy()

# 혹시 동일 (시장명_key, sido_key) 중복이 남으면 안전 처리
# - is_food_based/has_*는 max(OR)
# - item_diversity는 max
agg = {c: "first" for c in add_cols}
for c in add_cols:
    if c.startswith("has_") or c in ["is_food_based", "has_nonfood"]:
        agg[c] = "max"
    if c == "item_diversity":
        agg[c] = "max"

m6_keep = (
    m6_keep
    .groupby(["시장명_key", "sido_key"], as_index=False)
    .agg(agg)
)


In [169]:
mdf_merged = mdf.merge(
    m6_keep,
    on=["시장명_key", "sido_key"],
    how="left"
)

# 원하면 key 컬럼 제거
market_df = mdf_merged.drop(columns=["시장명_key", "sido_key"], errors="ignore")


In [129]:
cols_check = [
    'has_assoc', 'join_stores', 'delivery',
    'grocery', 'item_diversity', 'is_food_based',
    'has_nonfood', 'fresh_count', 'nonfood_count'
]

rows_with_nan = market_df[
    market_df[cols_check].isna().any(axis=1)
]

rows_with_nan


,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,지원여부,has_assoc,join_stores,delivery,grocery,item_diversity,is_food_based,has_nonfood,fresh_count,nonfood_count
9,가평잣고을시장,경기도 가평군 읍내리 405,경기도 가평군 가평읍 장터2길12 204호,경기도,가평군,Y,N,Y,Y,Y,...,1.0,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0
11,간성전통시장,강원도 고성군 간성읍 신안리 311-10,강원도 고성군 간성시장3길 7-2,강원특별자치도,고성군,Y,N,Y,N,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,갈산전통시장,충청남도 홍성군 갈산면 상촌리 240-16,충청남도 홍성군 갈산면 상촌로 29,충청남도,홍성군,Y,N,Y,N,Y,...,NaN,NaN,NaN,NaN,NaN,7.0,1.0,1.0,2.0,3.0
21,강릉동부시장,강원도 강릉시 옥천동 223,강원도 강릉시 옥천로 48,강원특별자치도,강릉시,Y,N,N,Y,Y,...,NaN,1.0,72.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
22,강릉서부시장,강원도 강릉시 용강동 29,강원도 강릉시 임영로 155번길 6,강원특별자치도,강릉시,Y,N,Y,Y,Y,...,NaN,1.0,53.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967,황지자유시장,강원도 태백시 통동 75-32,강원도 태백시 시장안길 18 - 1,강원특별자치도,태백시,Y,N,Y,N,Y,...,1.0,1.0,144.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
971,회인재래시장,충청북도 보은군 회인면 중앙리 59-15,충청북도 보은근 회인면 회인로 34,충청북도,보은군,Y,N,N,Y,N,...,NaN,NaN,NaN,0.0,0.0,9.0,1.0,1.0,3.0,4.0
972,횡성시장,강원도 횡성군 횡성읍 삼일로 4-2,강원도 횡성군 횡성읍 읍상리 277-3,강원특별자치도,횡성군,Y,N,Y,N,Y,...,NaN,1.0,125.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
976,흥덕전통시장,전라북도 고창군 흥덕면 흥덕리 239-1,전라북도 고창군 흥덕면 흥덕시장길 3,전라북도,고창군,Y,N,Y,N,Y,...,NaN,1.0,10.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


In [130]:
market_df[cols_check].isna().sum().sort_values(ascending=False)


item_diversity    103
nonfood_count     103
fresh_count       103
has_nonfood       103
is_food_based     103
has_assoc          98
join_stores        98
grocery            46
delivery           46
dtype: int64

In [133]:
market_df.columns

Index(['시장명', '지번주소', '도로명주소', '시도', '시군구', '아케이드 보유 여부', '엘리베이터_에스컬레이터_보유여부',
       '고객지원센터 보유 여부', '스프링쿨러 보유 여부', '화재감지기 보유여부', '유아놀이방_보유여부', '종합콜센터_보유여부',
       '고객휴게실_보유여부', '수유센터_보유여부', '물품보관함_보유여부', '자전거보관함_보유여부', '체육시설_보유여부',
       '간이 도서관_보유여부', '쇼핑카트_보유여부', '외국인 안내센터_보유여부', '고객동선통로_보유여부', '방송센터_보유여부',
       '문화교실_보유여부', '공동물류창고_보유여부', '시장전용 고객주차장_보유여부', '교육장_보유여부', '회의실_보유여부',
       '자동심장충격기_보유여부', '시장면적', '전체점포', '빈점포', '기타점포', '노점수', '점포상인', '종업원',
       '노점상인', '총시장상인', '지원금액', '지원여부', 'has_assoc', 'join_stores', 'delivery',
       'grocery', 'item_diversity', 'is_food_based', 'has_nonfood',
       'fresh_count', 'nonfood_count'],
      dtype='object')

In [172]:
zero_cols = [
    '지원금액', '지원여부', 'has_assoc', 'join_stores',
    'delivery', 'grocery'
]

market_df[zero_cols] = market_df[zero_cols].fillna(0)

market_df["market_item_type"] = (
    market_df["market_item_type"].fillna("미분류")
)

In [179]:
drop_cols = [
    'item_diversity','nonfood_count','fresh_count',
    'has_nonfood','is_food_based'
]

market_df = market_df.dropna(subset=drop_cols)


In [ ]:
market_df.shape

(876, 49)

In [181]:
market_df.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df.csv",
    index=False, encoding="utf-8-sig")

## 인구수 맵핑

In [182]:
import pandas as pd
import numpy as np
import re

market_df = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df.csv", encoding="utf-8-sig")

kikmix_df = pd.read_csv(r"D:\PP\BC\data\og\KIKmix_20230701.csv", encoding="utf-8-sig")
# kikmix_df columns: ['행정동코드','시도명','시군구명','읍면동명', ...]

pop_df = pd.read_csv(r"D:\PP\BC\data\og\지역별(행정동) 성별 연령별 주민등록 인구수_20231031.csv", encoding="cp949")


### market_df: 시도, 시군구 별 분리

In [183]:
sido_lists = market_df["시도"].dropna().unique()

# 1) 시도 -> df 형태로 저장
market_df_sido = {}

for sido in sido_lists:
    suffix = sido_map.get(sido)
    if suffix is None:
        print(f"⚠️ 매핑되지 않은 시도: {sido}")
        continue

    market_df_sido[f"{suffix}"] = market_df[market_df['시도'] == sido].copy()


# 2) 시도 -> 시군구 -> df 형태로 저장
market_df_sido_sigungu = {}

for sido in sido_lists:
    df_sido = market_df[market_df["시도"] == sido].copy()

    sigungu_lists = df_sido["시군구"].dropna().unique()

    market_df_sido_sigungu[sido] = {}
    for sigungu in sigungu_lists:
        market_df_sido_sigungu[sido][sigungu] = df_sido[df_sido["시군구"] == sigungu].copy()


In [184]:
for sido in sido_lists:
    suffix = sido_map.get(sido)
    print(f"{sido}: {market_df_sido[suffix].shape}")

울산광역시: (34, 49)
서울특별시: (137, 49)
충청북도: (32, 49)
부산광역시: (124, 49)
경상남도: (128, 49)
경상북도: (101, 49)
경기도: (91, 49)
인천광역시: (29, 49)
충청남도: (27, 49)
전라남도: (50, 49)
제주특별자치도: (13, 49)
대구광역시: (68, 49)
광주광역시: (17, 49)
대전광역시: (21, 49)
세종특별자치시: (4, 49)


In [187]:
for sido, sigungu_dict in market_df_sido_sigungu.items():
    for sigungu, df in sigungu_dict.items():
        print(f"{sido} - {sigungu}: {df.shape}")

울산광역시 - 남구: (8, 49)
울산광역시 - 울주군: (8, 49)
울산광역시 - 중구: (14, 49)
울산광역시 - 동구: (3, 49)
울산광역시 - 북구: (1, 49)
서울특별시 - 중구: (20, 49)
서울특별시 - 구로구: (4, 49)
서울특별시 - 강남구: (1, 49)
서울특별시 - 동대문구: (10, 49)
서울특별시 - 양천구: (5, 49)
서울특별시 - 강동구: (6, 49)
서울특별시 - 마포구: (5, 49)
서울특별시 - 노원구: (2, 49)
서울특별시 - 관악구: (10, 49)
서울특별시 - 광진구: (2, 49)
서울특별시 - 종로구: (5, 49)
서울특별시 - 성동구: (2, 49)
서울특별시 - 성북구: (5, 49)
서울특별시 - 강서구: (8, 49)
서울특별시 - 서초구: (1, 49)
서울특별시 - 동작구: (4, 49)
서울특별시 - 은평구: (6, 49)
서울특별시 - 영등포구: (4, 49)
서울특별시 - 금천구: (3, 49)
서울특별시 - 중랑구: (9, 49)
서울특별시 - 송파구: (6, 49)
서울특별시 - 용산구: (5, 49)
서울특별시 - 도봉구: (6, 49)
서울특별시 - 서대문구: (4, 49)
서울특별시 - 강북구: (4, 49)
충청북도 - 청주시: (9, 49)
충청북도 - 충주시: (5, 49)
충청북도 - 제천시: (8, 49)
충청북도 - 음성군: (3, 49)
충청북도 - 보은군: (3, 49)
충청북도 - 괴산군: (2, 49)
충청북도 - 옥천군: (1, 49)
충청북도 - 진천군: (1, 49)
부산광역시 - 사하구: (11, 49)
부산광역시 - 부산진구: (19, 49)
부산광역시 - 사상구: (10, 49)
부산광역시 - 연제구: (4, 49)
부산광역시 - 수영구: (13, 49)
부산광역시 - 금정구: (7, 49)
부산광역시 - 북구: (3, 49)
부산광역시 - 동래구: (10, 49)
부산광역시 - 중구: (12, 49)
부산광역시 - 기장군: (

### kikmix_df: 시도, 시군구 별 행정기관코드 시도(2자리) + 시군구(2자리) 코드

* 행정기관코드 시도, 시군구 필터링
    * 행정기관코드_시도: 행정기관코드 앞 2자리
    * 행정기관코드_시군구: 행정기관코드 앞 3~4자리
        * 시도(2자리) + 시군구(2자리)

    * 시도명 -> 시군구명 매칭
        * kikmix_df와 매칭
    * 시도명 먼저 매치 후 확인: market_df_sido 데이터에서 각 첫번째 리스트가 잘 맵핑됐는지
    * 다음으로 시군구명 매칭 후 확인: market_df_sido_sigungu 데이터에서 시군구까지 잘 맵핑됐는지 확인
    * 이걸로 시도명, 시군구명은 완벽히 매칭 가능

    * 따로 kikmix_df_sido_sigungu를 생성해서 market_df_sido_sigungu 처럼 분리해서 행정동기관, 동리명 등 저장


In [188]:
kikmix_df["행정동코드_시도"] = kikmix_df["행정동코드"].astype(str).str[:2].astype(int)

kikmix_df["행정동코드_시군구"] = kikmix_df["행정동코드"].astype(str).str[2:4].astype(int)

kikmix_df[["시도명", "행정동코드_시도", "행정동코드_시군구"]].drop_duplicates()

,시도명,행정동코드_시도,행정동코드_시군구
0,서울특별시,11,0
1,서울특별시,11,11
95,서울특별시,11,14
188,서울특별시,11,17
227,서울특별시,11,20
...,...,...,...
21391,강원특별자치도,51,79
21437,강원특별자치도,51,80
21503,강원특별자치도,51,81
21555,강원특별자치도,51,82


In [189]:
kikmix_df[["시도명", "행정동코드_시도"]].dropna().drop_duplicates()

,시도명,행정동코드_시도
0,서울특별시,11
769,부산광역시,26
1147,대구광역시,27
1652,인천광역시,28
2042,광주광역시,29
2317,대전광역시,30
2525,울산광역시,31
2751,세종특별자치시,36
2903,경기도,41
2904,북부출장소,41


In [190]:
sido_code = (
    kikmix_df[["시도명", "행정동코드_시도"]]
    .dropna()
    .drop_duplicates()
    .groupby("시도명")["행정동코드_시도"]
)

# 이상치(한 시도명에 코드가 2개 이상) 확인
bad = sido_code.nunique()
bad = bad[bad != 1]
if len(bad) > 0:
    print("⚠️ 시도명에 코드가 2개 이상 매칭되는 케이스:")
    print(bad)


In [191]:
sido_to_code = sido_code.first().to_dict()

In [192]:
sido_to_code

{'강원특별자치도': 51,
 '경기도': 41,
 '경상남도': 48,
 '경상북도': 47,
 '광주광역시': 29,
 '대구광역시': 27,
 '대전광역시': 30,
 '동해출장소': 51,
 '부산광역시': 26,
 '북부출장소': 41,
 '서울특별시': 11,
 '세종특별자치시': 36,
 '울산광역시': 31,
 '인천광역시': 28,
 '전라남도': 46,
 '전라북도': 45,
 '제주특별자치도': 50,
 '충청남도': 44,
 '충청북도': 43}

In [193]:
sigungu_code = (
    kikmix_df[["시도명", "시군구명", "행정동코드_시도", "행정동코드_시군구"]]
    .dropna(subset=["시도명", "시군구명"])
    .drop_duplicates()
    .groupby(["시도명", "시군구명"])
)

# 이상치 체크: (시도명, 시군구명)인데 코드가 여러 개면 경고
bad_sido = sigungu_code["행정동코드_시도"].nunique()
bad_sig  = sigungu_code["행정동코드_시군구"].nunique()

bad_pairs = pd.concat([
    bad_sido.rename("n_sido"),
    bad_sig.rename("n_sigungu")
], axis=1)

bad_pairs = bad_pairs[(bad_pairs["n_sido"] != 1) | (bad_pairs["n_sigungu"] != 1)]
if len(bad_pairs) > 0:
    print("⚠️ (시도명, 시군구명) 매핑이 1:1이 아닌 케이스가 있음:")
    print(bad_pairs.head(30))



In [194]:
# 정상 매핑 dict: (시도명, 시군구명) -> (code2, code_sg)
sido_sigungu_to_code = {
    idx: (row["행정동코드_시도"], row["행정동코드_시군구"])
    for idx, row in sigungu_code[["행정동코드_시도", "행정동코드_시군구"]].first().iterrows()
}

In [195]:
sido_sigungu_to_code

{('강원특별자치도', '강릉시'): (np.int64(51), np.int64(15)),
 ('강원특별자치도', '고성군'): (np.int64(51), np.int64(82)),
 ('강원특별자치도', '동해시'): (np.int64(51), np.int64(17)),
 ('강원특별자치도', '삼척시'): (np.int64(51), np.int64(23)),
 ('강원특별자치도', '속초시'): (np.int64(51), np.int64(21)),
 ('강원특별자치도', '양구군'): (np.int64(51), np.int64(80)),
 ('강원특별자치도', '양양군'): (np.int64(51), np.int64(83)),
 ('강원특별자치도', '영월군'): (np.int64(51), np.int64(75)),
 ('강원특별자치도', '원주시'): (np.int64(51), np.int64(13)),
 ('강원특별자치도', '인제군'): (np.int64(51), np.int64(81)),
 ('강원특별자치도', '정선군'): (np.int64(51), np.int64(77)),
 ('강원특별자치도', '철원군'): (np.int64(51), np.int64(78)),
 ('강원특별자치도', '춘천시'): (np.int64(51), np.int64(11)),
 ('강원특별자치도', '태백시'): (np.int64(51), np.int64(19)),
 ('강원특별자치도', '평창군'): (np.int64(51), np.int64(76)),
 ('강원특별자치도', '홍천군'): (np.int64(51), np.int64(72)),
 ('강원특별자치도', '화천군'): (np.int64(51), np.int64(79)),
 ('강원특별자치도', '횡성군'): (np.int64(51), np.int64(73)),
 ('경기도', '가평군'): (np.int64(41), np.int64(82)),
 ('경기도', '고양시'): (np.int64(41), np.

군위군: kikmix_df에 없음

세종시: kikmix_df의 세종특별자치시의 시군도명은 비어있음

In [196]:
kikmix_df_sido_sigungu = {}

for sido, sigungu_dict in market_df_sido_sigungu.items():
    kikmix_df_sido_sigungu[sido] = {}

    for sigungu in sigungu_dict.keys():
        code_pair = sido_sigungu_to_code.get((sido, sigungu))

        # 예외 처리: 세종특별자치시 - 세종시
        if code_pair is None and (sido == "세종특별자치시"):
            kikmix_df_sido_sigungu[sido][sigungu] = kikmix_df[
                kikmix_df["시도명"] == "세종특별자치시"
            ].copy()
            print(f"{sido} - {sigungu} 처리 완료")
            continue
        
        # 일반 케이스
        if code_pair is None:
            print(f"⚠️ 매칭 실패: {sido} - {sigungu}")
            continue

        code2, code_sg = code_pair

        kikmix_df_sido_sigungu[sido][sigungu] = kikmix_df[
            (kikmix_df["행정동코드_시도"] == code2) &
            (kikmix_df["행정동코드_시군구"] == code_sg)
        ].copy()


세종특별자치시 - 세종시 처리 완료


In [197]:
kikmix_df_sido_sigungu['서울특별시']['광진구']

,행정동코드,시도명,시군구명,읍면동명,법정동코드,동리명,생성일자,말소일자,행정동코드_시도,행정동코드_시군구
251,1121500000,서울특별시,광진구,NaN,1121500000,광진구,19950301,NaN,11,21
252,1121571000,서울특별시,광진구,화양동,1121510700,화양동,19950301,NaN,11,21
253,1121573000,서울특별시,광진구,군자동,1121510900,군자동,19950301,NaN,11,21
254,1121574000,서울특별시,광진구,중곡제1동,1121510100,중곡동,19950301,NaN,11,21
255,1121575000,서울특별시,광진구,중곡제2동,1121510100,중곡동,19950301,NaN,11,21
256,1121576000,서울특별시,광진구,중곡제3동,1121510100,중곡동,19950301,NaN,11,21
257,1121577000,서울특별시,광진구,중곡제4동,1121510100,중곡동,19950301,NaN,11,21
258,1121578000,서울특별시,광진구,능동,1121510200,능동,19950301,NaN,11,21
259,1121581000,서울특별시,광진구,광장동,1121510400,광장동,19950301,NaN,11,21
260,1121582000,서울특별시,광진구,자양제1동,1121510500,자양동,19950301,NaN,11,21


### market_df & kikmix_df 매칭: 시도, 시군구 별

1) 시장 데이터에서 “동 후보” 만들기

    *  지번주소: 옥천동 223처럼 동/읍/면/리

    * 도로명주소: 옥천로 48이라 동이 빠지는 경우가 많아서 보조로만 씀

    * 마지막에 가까운 토큰 중 동/읍/면/리로 끝나는 단어를 뽑아서 dong_guess로 저장

2) kikmix에서 매칭용 “사전” 만들기

    * (해당 시도-시군구 subset 안에서)

        읍면동명 유니크 set

        동리명 유니크 set

    * 그리고 “동/읍/면/리 → 행정동코드”로 맵핑 dict 만들기

    * 같은 이름이 여러 코드로 매핑되면(동리명에서 흔함) 모호로 처리하고 일단 NA

3) 매칭 우선순위

    * dong_guess가 읍면동명에 유일하게 존재하면 → 그 읍면동명 코드로

    * 아니면 dong_guess가 동리명에 유일하게 존재하면 → 그 동리명 코드로

    * 그래도 안 되면 NA로 두고 “매칭 실패/모호” 로그에 남김

#### 행정동

In [200]:
import re
import numpy as np
import pandas as pd

# 1) 지번주소에서 동/읍/면/리 후보 추출
def extract_dong_from_jibun(addr: str):
    if pd.isna(addr):
        return np.nan
    addr = str(addr).strip()
    # 공백 기준 토큰화 후, 뒤에서부터 동/읍/면/리로 끝나는 토큰 찾기
    tokens = addr.split()
    for tok in reversed(tokens):
        # 예: 옥천동, 주문진읍, 성산면, 주문리
        if re.search(r"(동([0-9]+가)?|읍|면|리|로([0-9]+가)?|길|대로)$", tok):
            # '옥천동 223-1' 같은 경우는 tok가 '옥천동'이라 OK
            return tok
    return np.nan


# 2) (해당 시도-시군구 내부에서) "이름 -> 행정동코드" 매핑 dict 만들기
def build_name_to_code_maps(kikmix_sub: pd.DataFrame):
    # 읍면동명: 보통 행정단위 (상위)
    e_map = (
        kikmix_sub.dropna(subset=["읍면동명"])
        .groupby("읍면동명")["행정동코드"]
        .nunique()
    )
    # 유일 매핑만 사용
    e_unique_names = e_map[e_map == 1].index
    e_name_to_code = (
        kikmix_sub[kikmix_sub["읍면동명"].isin(e_unique_names)]
        .dropna(subset=["읍면동명"])
        .groupby("읍면동명")["행정동코드"]
        .first()
        .to_dict()
    )

    # 동리명: 법정단위 (하위) - 중복이 더 흔해서 유일한 것만 쓰는 게 안전
    d_map = (
        kikmix_sub.dropna(subset=["동리명"])
        .groupby("동리명")["행정동코드"]
        .nunique()
    )
    d_unique_names = d_map[d_map == 1].index
    d_name_to_code = (
        kikmix_sub[kikmix_sub["동리명"].isin(d_unique_names)]
        .dropna(subset=["동리명"])
        .groupby("동리명")["행정동코드"]
        .first()
        .to_dict()
    )

    return e_name_to_code, d_name_to_code


# 3) 시도-시군구 단위로 market_sub에 행정동코드 매핑
def match_admin_code_within_region(market_sub: pd.DataFrame, kikmix_sub: pd.DataFrame):
    market_sub = market_sub.copy()
    kikmix_sub = kikmix_sub.copy()

    # (A) 시장 데이터에서 동 후보 추출
    market_sub["dong_guess"] = market_sub["지번주소"].apply(extract_dong_from_jibun)
    market_sub["dong_guess_je"] = np.nan

    # (B) kikmix에서 매칭 dict 만들기 (유일 매핑만)
    e_name_to_code, d_name_to_code = build_name_to_code_maps(kikmix_sub)

    # (C) 우선순위 매칭
    def _match_one(dong, dong_je):
        if pd.isna(dong):
            return np.nan, "no_dong_guess"

        # 1) 읍면동명 유일 매칭 (원본)
        if dong in e_name_to_code:
            return e_name_to_code[dong], "match_eupmyeondong"

        # 1-1) 읍면동명 유일 매칭 (제 변형)
        if (not pd.isna(dong_je)) and (dong_je in e_name_to_code):
            return e_name_to_code[dong_je], "match_eupmyeondong_je"

        # 2) 동리명 유일 매칭 (원본)
        if dong in d_name_to_code:
            return d_name_to_code[dong], "match_dongri"

        # 2-1) 동리명 유일 매칭 (제 변형)
        if (not pd.isna(dong_je)) and (dong_je in d_name_to_code):
            return d_name_to_code[dong_je], "match_dongri_je"

        return np.nan, "no_match"

    #matched = market_sub["dong_guess"].apply(_match_one)
    matched = market_sub.apply(lambda r: _match_one(r["dong_guess"], r["dong_guess_je"]), axis=1)
    market_sub["행정기관코드"] = matched.apply(lambda x: x[0])
    market_sub["match_status"] = matched.apply(lambda x: x[1])
    market_sub["행정기관코드"] = matched.apply(lambda x: x[0])
    return market_sub


In [201]:
market_df_sido_sigungu_matched = {}

for sido, sigungu_dict in market_df_sido_sigungu.items():
    market_df_sido_sigungu_matched[sido] = {}

    for sigungu, market_sub in sigungu_dict.items():
        kikmix_sub = kikmix_df_sido_sigungu.get(sido, {}).get(sigungu)

        if kikmix_sub is None:
            print(f"⚠️ kikmix 없음: {sido} - {sigungu}")
            market_df_sido_sigungu_matched[sido][sigungu] = market_sub.copy()
            continue

        market_df_sido_sigungu_matched[sido][sigungu] = match_admin_code_within_region(
            market_sub=market_sub,
            kikmix_sub=kikmix_sub
        )


In [202]:
# kikmix에 없는거 삭제
market_df_sido_sigungu_matched.get("경상북도", {}).pop("군위군", None)

In [203]:
market_df_sido_sigungu_matched['울산광역시']['남구']

,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,item_diversity,is_food_based,has_nonfood,fresh_count,nonfood_count,market_item_type,dong_guess,dong_guess_je,행정기관코드,match_status
0,(주)신정시장,울산광역시 남구 신정동 630-1,울산광역시 남구 월평로47번길 7,울산광역시,남구,Y,N,Y,N,Y,...,7.0,1.0,1.0,3.0,2.0,복합형(생활+비식품),신정동,NaN,NaN,no_match
462,수암상가시장,울산광역시 남구 야음동 697-9,울산광역시 남구 수암로 128번길 12,울산광역시,남구,Y,N,Y,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),야음동,NaN,NaN,no_match
463,수암종합시장,울산광역시 남구 야음동 698-1,울산광역시 남구 수암로 124 야음동,울산광역시,남구,Y,N,Y,N,Y,...,5.0,1.0,1.0,1.0,3.0,복합형(생활+비식품),야음동,NaN,NaN,no_match
494,신정상가시장,울산광역시 남구 신정동 634-10,울산광역시 남구 중앙로241번길 14-1,울산광역시,남구,Y,N,Y,N,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),신정동,NaN,NaN,no_match
496,신정평화시장,울산광역시 남구 신정동 1600-3 신정평화시장,울산광역시 남구 봉월로 67번길 17,울산광역시,남구,N,N,Y,Y,Y,...,6.0,1.0,1.0,2.0,2.0,복합형(생활+비식품),신정동,NaN,NaN,no_match
531,야음상가시장,울산광역시 남구 야음동 815-10,울산광역시 남구 수암로 244-6,울산광역시,남구,Y,N,Y,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),야음동,NaN,NaN,no_match
630,울산번개시장,울산광역시 남구 야음동 375-55,울산광역시 남구 신선로184번길 46-1 야음동,울산광역시,남구,Y,N,Y,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),야음동,NaN,NaN,no_match
631,울산산업공구월드,울산광역시 남구 삼산동 1591-4 울산산업공구월드,울산광역시 남구 갈밭로 24 삼산동,울산광역시,남구,N,Y,Y,Y,Y,...,1.0,0.0,1.0,0.0,1.0,서비스형,삼산동,NaN,3.114057e+09,match_eupmyeondong


In [204]:
kikmix_df_sido_sigungu['울산광역시']['북구']

,행정동코드,시도명,시군구명,읍면동명,법정동코드,동리명,생성일자,말소일자,행정동코드_시도,행정동코드_시군구
2590,3120000000,울산광역시,북구,NaN,3120000000,북구,19970715,NaN,31,20
2591,3120051000,울산광역시,북구,농소1동,3120010100,창평동,19970715,NaN,31,20
2592,3120051000,울산광역시,북구,농소1동,3120010200,호계동,19970715,NaN,31,20
2593,3120051000,울산광역시,북구,농소1동,3120010300,매곡동,19970715,NaN,31,20
2594,3120051000,울산광역시,북구,농소1동,3120010500,신천동,19970715,NaN,31,20
2595,3120052000,울산광역시,북구,농소2동,3120010300,매곡동,19970715,NaN,31,20
2596,3120052000,울산광역시,북구,농소2동,3120010500,신천동,19970715,NaN,31,20
2597,3120052000,울산광역시,북구,농소2동,3120010600,중산동,19970715,NaN,31,20
2598,3120053000,울산광역시,북구,농소3동,3120010400,가대동,19970715,NaN,31,20
2599,3120053000,울산광역시,북구,농소3동,3120010700,상안동,19970715,NaN,31,20


In [205]:
market_df_sido_sigungu_matched['부산광역시']['남구']

,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,item_diversity,is_food_based,has_nonfood,fresh_count,nonfood_count,market_item_type,dong_guess,dong_guess_je,행정기관코드,match_status
118,남광시장,부산광역시 남구 대연4동1170-7,부산광역시 남구 홍곡로 359,부산광역시,남구,N,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),NaN,NaN,NaN,no_dong_guess
175,대연시장,부산광역시 남구 대연5동 1760-1,부산광역시 남구 못골로 56,부산광역시,남구,N,N,N,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),대연5동,NaN,NaN,no_match
269,못골골목시장,부산광역시 남구 대연5동 1416-23,부산광역시 남구 못골로 66-1,부산광역시,남구,N,N,Y,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),대연5동,NaN,NaN,no_match
617,용호1동골목시장,부산광역시 남구 용회동 215-14번지,부산광역시 남구 용호로106번길 56,부산광역시,남구,N,N,N,N,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용회동,NaN,NaN,no_match
618,용호골목시장,부산광역시 남구 용호동 494-19,부산광역시 남구 용호로 177번길5,부산광역시,남구,Y,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
619,용호삼성시장,부산광역시 남구 용호동 128-1,부산광역시 남구 용호로 106번길 40,부산광역시,남구,N,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
620,용호시장,부산광역시 남구 용호동 516-10,부산광역시 남구 동명로 152번길 93,부산광역시,남구,Y,N,Y,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
626,우암골목시장,부산광역시 남구 우암동 226-17,부산광역시 남구 장고개로 9-1,부산광역시,남구,N,N,N,N,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),우암동,NaN,2.629064e+09,match_eupmyeondong


In [206]:
kikmix_df_sido_sigungu['서울특별시']['중구']

,행정동코드,시도명,시군구명,읍면동명,법정동코드,동리명,생성일자,말소일자,행정동코드_시도,행정동코드_시군구
95,1114000000,서울특별시,중구,NaN,1114000000,중구,19880423,NaN,11,14
96,1114052000,서울특별시,중구,소공동,1114011100,소공동,19880423,NaN,11,14
97,1114052000,서울특별시,중구,소공동,1114011300,북창동,19880423,NaN,11,14
98,1114052000,서울특별시,중구,소공동,1114011400,태평로2가,19880423,NaN,11,14
99,1114052000,서울특별시,중구,소공동,1114011500,남대문로2가,19880423,NaN,11,14
...,...,...,...,...,...,...,...,...,...,...
183,1114067000,서울특별시,중구,황학동,1114016500,황학동,19880423,NaN,11,14
184,1114068000,서울특별시,중구,중림동,1114017100,중림동,19880423,NaN,11,14
185,1114068000,서울특별시,중구,중림동,1114017200,의주로2가,19880423,NaN,11,14
186,1114068000,서울특별시,중구,중림동,1114017300,만리동1가,19880423,NaN,11,14


### 매칭 성공률

In [207]:
for sido, sigungu_dict in market_df_sido_sigungu_matched.items():
    for sigungu, df in sigungu_dict.items():
        if "match_status" not in df.columns:
            continue
        rate = (df["행정기관코드"].notna().mean() * 100)
        print(f"{sido} - {sigungu}: matched {rate:.1f}%  (n={len(df)})")


울산광역시 - 남구: matched 12.5%  (n=8)
울산광역시 - 울주군: matched 87.5%  (n=8)
울산광역시 - 중구: matched 85.7%  (n=14)
울산광역시 - 동구: matched 33.3%  (n=3)
울산광역시 - 북구: matched 100.0%  (n=1)
서울특별시 - 중구: matched 95.0%  (n=20)
서울특별시 - 구로구: matched 25.0%  (n=4)
서울특별시 - 강남구: matched 100.0%  (n=1)
서울특별시 - 동대문구: matched 80.0%  (n=10)
서울특별시 - 양천구: matched 20.0%  (n=5)
서울특별시 - 강동구: matched 0.0%  (n=6)
서울특별시 - 마포구: matched 60.0%  (n=5)
서울특별시 - 노원구: matched 50.0%  (n=2)
서울특별시 - 관악구: matched 70.0%  (n=10)
서울특별시 - 광진구: matched 50.0%  (n=2)
서울특별시 - 종로구: matched 100.0%  (n=5)
서울특별시 - 성동구: matched 50.0%  (n=2)
서울특별시 - 성북구: matched 20.0%  (n=5)
서울특별시 - 강서구: matched 12.5%  (n=8)
서울특별시 - 서초구: matched 0.0%  (n=1)
서울특별시 - 동작구: matched 0.0%  (n=4)
서울특별시 - 은평구: matched 50.0%  (n=6)
서울특별시 - 영등포구: matched 75.0%  (n=4)
서울특별시 - 금천구: matched 0.0%  (n=3)
서울특별시 - 중랑구: matched 0.0%  (n=9)
서울특별시 - 송파구: matched 16.7%  (n=6)
서울특별시 - 용산구: matched 60.0%  (n=5)
서울특별시 - 도봉구: matched 0.0%  (n=6)
서울특별시 - 서대문구: matched 25.0%  (n=4)
서울특별시 - 강북구: ma

In [208]:
import pandas as pd

def calc_overall_match_rate(market_df_sido_sigungu_matched):
    total_rows = 0
    matched_rows = 0

    for sido, sigungu_dict in market_df_sido_sigungu_matched.items():
        for sigungu, df in sigungu_dict.items():
            if "행정기관코드" not in df.columns:
                continue

            total_rows += len(df)
            matched_rows += df["행정기관코드"].notna().sum()

    match_rate = matched_rows / total_rows if total_rows > 0 else 0

    return {
        "total_rows": total_rows,
        "matched_rows": matched_rows,
        "match_rate": match_rate
    }

def calc_match_rate_by_sido(market_df_sido_sigungu_matched):
    rows = []

    for sido, sigungu_dict in market_df_sido_sigungu_matched.items():
        total = 0
        matched = 0

        for df in sigungu_dict.values():
            if "행정기관코드" not in df.columns:
                continue
            total += len(df)
            matched += df["행정기관코드"].notna().sum()

        if total > 0:
            rows.append({
                "시도": sido,
                "전체시장수": total,
                "매칭성공수": matched,
                "매칭성공률(%)": round(matched / total * 100, 2)
            })

    return pd.DataFrame(rows).sort_values("매칭성공률(%)", ascending=False)


In [209]:
result = calc_overall_match_rate(market_df_sido_sigungu_matched)

print(f"전체 시장 수: {result['total_rows']}")
print(f"행정동코드 매칭 성공: {result['matched_rows']}")
print(f"전체 매칭 성공률: {result['match_rate']*100:.2f}%")


전체 시장 수: 876
행정동코드 매칭 성공: 539
전체 매칭 성공률: 61.53%


In [210]:
calc_match_rate_by_sido(market_df_sido_sigungu_matched)

,시도,전체시장수,매칭성공수,매칭성공률(%)
14,세종특별자치시,4,4,100.00
10,제주특별자치도,13,12,92.31
2,충청북도,32,29,90.62
5,경상북도,101,91,90.10
8,충청남도,27,24,88.89
9,전라남도,50,43,86.00
4,경상남도,128,105,82.03
13,대전광역시,21,17,80.95
0,울산광역시,34,22,64.71
12,광주광역시,17,10,58.82


In [211]:
kikmix_df_sido_sigungu['부산광역시']['남구']

,행정동코드,시도명,시군구명,읍면동명,법정동코드,동리명,생성일자,말소일자,행정동코드_시도,행정동코드_시군구
928,2629000000,부산광역시,남구,NaN,2629000000,남구,19950101,NaN,26,29
929,2629051000,부산광역시,남구,대연제1동,2629010600,대연동,19950101,NaN,26,29
930,2629053000,부산광역시,남구,대연제3동,2629010600,대연동,19950101,NaN,26,29
931,2629054000,부산광역시,남구,대연제4동,2629010600,대연동,19950101,NaN,26,29
932,2629055000,부산광역시,남구,대연제5동,2629010600,대연동,19950101,NaN,26,29
933,2629056000,부산광역시,남구,대연제6동,2629010600,대연동,19950101,NaN,26,29
934,2629057000,부산광역시,남구,용호제1동,2629010700,용호동,19950101,NaN,26,29
935,2629058000,부산광역시,남구,용호제2동,2629010700,용호동,19950101,NaN,26,29
936,2629059000,부산광역시,남구,용호제3동,2629010700,용호동,19950101,NaN,26,29
937,2629060000,부산광역시,남구,용호제4동,2629010700,용호동,19950101,NaN,26,29


In [212]:
market_df_sido_sigungu_matched['부산광역시']['남구']

,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,item_diversity,is_food_based,has_nonfood,fresh_count,nonfood_count,market_item_type,dong_guess,dong_guess_je,행정기관코드,match_status
118,남광시장,부산광역시 남구 대연4동1170-7,부산광역시 남구 홍곡로 359,부산광역시,남구,N,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),NaN,NaN,NaN,no_dong_guess
175,대연시장,부산광역시 남구 대연5동 1760-1,부산광역시 남구 못골로 56,부산광역시,남구,N,N,N,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),대연5동,NaN,NaN,no_match
269,못골골목시장,부산광역시 남구 대연5동 1416-23,부산광역시 남구 못골로 66-1,부산광역시,남구,N,N,Y,Y,Y,...,9.0,1.0,1.0,3.0,4.0,복합형(생활+비식품),대연5동,NaN,NaN,no_match
617,용호1동골목시장,부산광역시 남구 용회동 215-14번지,부산광역시 남구 용호로106번길 56,부산광역시,남구,N,N,N,N,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용회동,NaN,NaN,no_match
618,용호골목시장,부산광역시 남구 용호동 494-19,부산광역시 남구 용호로 177번길5,부산광역시,남구,Y,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
619,용호삼성시장,부산광역시 남구 용호동 128-1,부산광역시 남구 용호로 106번길 40,부산광역시,남구,N,N,N,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
620,용호시장,부산광역시 남구 용호동 516-10,부산광역시 남구 동명로 152번길 93,부산광역시,남구,Y,N,Y,Y,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),용호동,NaN,NaN,no_match
626,우암골목시장,부산광역시 남구 우암동 226-17,부산광역시 남구 장고개로 9-1,부산광역시,남구,N,N,N,N,Y,...,8.0,1.0,1.0,3.0,3.0,복합형(생활+비식품),우암동,NaN,2.629064e+09,match_eupmyeondong


---

In [213]:
import numpy as np
import pandas as pd

def fill_admin_code_by_address_priority(market_sub: pd.DataFrame,
                                       kikmix_sub: pd.DataFrame,
                                       addr_col="지번주소",
                                       out_col="행정기관코드",
                                       status_col="match_status",
                                       target_status="no_match",
                                       debug=False) -> pd.DataFrame:
    """
    우선순위:
    1) 지번주소에 읍면동명 '전체' 포함 -> 후보들 중 행정동코드 min
    2) 없으면 동리명 '전체' 포함 -> 후보들 중 행정동코드 min
    3) 없으면 prefix2(읍면동명[:2] or 동리명[:2]) 포함 후보 -> 후보들 중 행정동코드 min
    """

    if kikmix_sub is None or len(kikmix_sub) == 0 or market_sub is None or len(market_sub) == 0:
        return market_sub

    # 0) kikmix: 읍면동명 NaN 아닌 것만 사용
    kk = kikmix_sub.copy()
    kk = kk[kk["읍면동명"].notna()].copy()
    if len(kk) == 0:
        return market_sub

    # 문자열 정리
    for c in ["읍면동명", "동리명"]:
        if c in kk.columns:
            kk[c] = kk[c].astype(str).str.strip()
    kk["행정동코드"] = pd.to_numeric(kk["행정동코드"], errors="coerce")

    # prefix2 컬럼 생성
    kk["읍면동명_2"] = kk["읍면동명"].str[:2]
    kk["동리명_2"] = kk["동리명"].astype(str).str[:2] if "동리명" in kk.columns else ""

    # 1) market 대상: no_match & 행정기관코드 비어있는 것만
    out = market_sub.copy()
    cond = out[status_col].eq(target_status)
    if out_col in out.columns:
        cond = cond & (out[out_col].isna())
    else:
        out[out_col] = np.nan

    idxs = out.index[cond].tolist()
    if debug:
        print(f"[DEBUG] target rows: {len(idxs)} / {len(out)}")

    # 2) 매칭 함수 (주소 1개에 대해 코드 1개 반환)
    def pick_code(addr: str):
        if pd.isna(addr):
            return np.nan
        addr = str(addr).strip()
        if addr == "":
            return np.nan

        # (A) 읍면동명 전체 포함 후보
        eup_hit = kk[kk["읍면동명"].apply(lambda x: x and x != "nan" and x in addr)]
        if len(eup_hit) > 0:
            return eup_hit["행정동코드"].min()

        # (B) 동리명 전체 포함 후보
        if "동리명" in kk.columns:
            dong_hit = kk[kk["동리명"].apply(lambda x: x and x != "nan" and x in addr)]
            if len(dong_hit) > 0:
                return dong_hit["행정동코드"].min()

        # (C) prefix2 포함 후보 (읍면동명[:2] 또는 동리명[:2])
        # 빈 prefix 제거
        cand_codes = []

        eup2 = kk["읍면동명_2"].dropna().unique().tolist()
        eup2 = [p for p in eup2 if isinstance(p, str) and p.strip() and p != "na" and p != "nan"]
        for p in eup2:
            if p in addr:
                cand_codes.append(kk.loc[kk["읍면동명_2"] == p, "행정동코드"].min())

        if "동리명_2" in kk.columns:
            dong2 = kk["동리명_2"].dropna().unique().tolist()
            dong2 = [p for p in dong2 if isinstance(p, str) and p.strip() and p != "na" and p != "nan"]
            for p in dong2:
                if p in addr:
                    cand_codes.append(kk.loc[kk["동리명_2"] == p, "행정동코드"].min())

        # cand_codes 안에 여러 개가 있으면 그 중 최소
        cand_codes = [c for c in cand_codes if pd.notna(c)]
        if len(cand_codes) > 0:
            return min(cand_codes)

        return np.nan

    # 3) 적용
    mapped = 0
    for i in idxs:
        code = pick_code(out.at[i, addr_col])
        if pd.notna(code):
            out.at[i, out_col] = int(code)
            mapped += 1

    if debug:
        print(f"[DEBUG] mapped: {mapped}")

    return out


def apply_fill_to_nested_dict(market_dict: dict, kikmix_dict: dict, debug=False):
    """
    market_dict[sido][sigungu] DataFrame에 대해,
    kikmix_dict[sido][sigungu]를 이용해 no_match 행정기관코드 채움.
    """
    for sido, sigungu_dict in market_dict.items():
        for sigungu, market_sub in sigungu_dict.items():
            kikmix_sub = kikmix_dict.get(sido, {}).get(sigungu)
            if kikmix_sub is None:
                continue

            market_dict[sido][sigungu] = fill_admin_code_by_address_priority(
                market_sub=market_sub,
                kikmix_sub=kikmix_sub,
                debug=debug
            )
    return market_dict


In [214]:
market_df_sido_sigungu_matched = apply_fill_to_nested_dict(
    market_df_sido_sigungu_matched,
    kikmix_df_sido_sigungu,
    debug=True
)


[DEBUG] target rows: 7 / 8
[DEBUG] mapped: 7
[DEBUG] target rows: 0 / 8
[DEBUG] mapped: 0
[DEBUG] target rows: 2 / 14
[DEBUG] mapped: 2
[DEBUG] target rows: 2 / 3
[DEBUG] mapped: 2
[DEBUG] target rows: 0 / 1
[DEBUG] mapped: 0
[DEBUG] target rows: 1 / 20
[DEBUG] mapped: 1
[DEBUG] target rows: 2 / 4
[DEBUG] mapped: 2
[DEBUG] target rows: 0 / 1
[DEBUG] mapped: 0
[DEBUG] target rows: 2 / 10
[DEBUG] mapped: 2
[DEBUG] target rows: 4 / 5
[DEBUG] mapped: 4
[DEBUG] target rows: 6 / 6
[DEBUG] mapped: 6
[DEBUG] target rows: 2 / 5
[DEBUG] mapped: 2
[DEBUG] target rows: 1 / 2
[DEBUG] mapped: 1
[DEBUG] target rows: 3 / 10
[DEBUG] mapped: 3
[DEBUG] target rows: 1 / 2
[DEBUG] mapped: 1
[DEBUG] target rows: 0 / 5
[DEBUG] mapped: 0
[DEBUG] target rows: 1 / 2
[DEBUG] mapped: 1
[DEBUG] target rows: 4 / 5
[DEBUG] mapped: 4
[DEBUG] target rows: 7 / 8
[DEBUG] mapped: 7
[DEBUG] target rows: 1 / 1
[DEBUG] mapped: 1
[DEBUG] target rows: 4 / 4
[DEBUG] mapped: 4
[DEBUG] target rows: 3 / 6
[DEBUG] mapped: 3
[DEBUG

In [215]:
kikmix_df_sido_sigungu['부산광역시']['북구']

,행정동코드,시도명,시군구명,읍면동명,법정동코드,동리명,생성일자,말소일자,행정동코드_시도,행정동코드_시군구
946,2632000000,부산광역시,북구,NaN,2632000000,북구,19950101,NaN,26,32
947,2632051000,부산광역시,북구,구포제1동,2632010500,구포동,19950101,NaN,26,32
948,2632052000,부산광역시,북구,구포제2동,2632010500,구포동,19950101,NaN,26,32
949,2632052100,부산광역시,북구,구포제3동,2632010500,구포동,19950101,NaN,26,32
950,2632053000,부산광역시,북구,금곡동,2632010100,금곡동,19950101,NaN,26,32
951,2632054100,부산광역시,북구,화명제1동,2632010200,화명동,20030701,NaN,26,32
952,2632054200,부산광역시,북구,화명제2동,2632010200,화명동,20030701,NaN,26,32
953,2632054300,부산광역시,북구,화명제3동,2632010200,화명동,20031222,NaN,26,32
954,2632055000,부산광역시,북구,덕천제1동,2632010400,덕천동,19950101,NaN,26,32
955,2632056000,부산광역시,북구,덕천제2동,2632010400,덕천동,19950101,NaN,26,32


In [2]:
import numpy as np
np.exp(-0.647)

np.float64(0.5236142656482635)

In [216]:
market_df_sido_sigungu_matched['부산광역시']['북구']

,시장명,지번주소,도로명주소,시도,시군구,아케이드 보유 여부,엘리베이터_에스컬레이터_보유여부,고객지원센터 보유 여부,스프링쿨러 보유 여부,화재감지기 보유여부,...,item_diversity,is_food_based,has_nonfood,fresh_count,nonfood_count,market_item_type,dong_guess,dong_guess_je,행정기관코드,match_status
94,구포축산물도매시장,부산광역시 북구 구포2동 1188-3번지,부산광역시 북구 사상로583번길 25,부산광역시,북구,Y,N,N,N,Y,...,1.0,1.0,0.0,1.0,0.0,복합형(생활+비식품),구포2동,NaN,2.632051e+09,no_match
190,덕천시장,부산광역시 북구 덕천동 411-5,부산광역시 북구 덕천동 411-5,부산광역시,북구,Y,N,Y,Y,Y,...,2.0,1.0,0.0,0.0,0.0,먹거리/외식형,덕천동,NaN,2.632055e+09,no_match
244,만덕시장,부산광역시 북구 만덕1동 829,부산광역시 북구 만덕1로24번길 8,부산광역시,북구,Y,N,Y,N,Y,...,3.0,1.0,0.0,1.0,0.0,먹거리/외식형,만덕1동,NaN,2.632057e+09,no_match


In [217]:
import pickle

with open(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_sido_sigungu_matched.pkl", "wb") as f:
    pickle.dump(market_df_sido_sigungu_matched, f)



In [218]:
# 불러오기
with open(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_sido_sigungu_matched.pkl", "rb") as f:
    market_df_sido_sigungu_matched = pickle.load(f)


In [219]:
# dict → 하나의 DataFrame
market_df_all = pd.concat(
    [
        df
        for sigungu_dict in market_df_sido_sigungu_matched.values()
        for df in sigungu_dict.values()
    ],
    ignore_index=True
)


In [223]:
market_df_all[market_df_all['행정기관코드'].isna()].shape[0] / market_df_all.shape[0]

0.0410958904109589

In [221]:
market_df_all.to_csv(
    r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_2.csv",
    index=False,
    encoding="utf-8-sig"
)

In [222]:
market_df_all = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_2.csv",
    encoding="utf-8-sig"
)